In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("C:/Users/vikasa s y/Downloads/web-scraping-quotes-analysis/data/raw/quotes_raw.csv")

In [12]:
df.shape

(100, 3)

In [5]:
df.head(10)

,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"['change', 'deep-thoughts', 'thinking', 'world']"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"['abilities', 'choices']"
2,“There are only two ways to live your life. On...,Albert Einstein,"['inspirational', 'life', 'live', 'miracle', '..."
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"['aliteracy', 'books', 'classic', 'humor']"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"['be-yourself', 'inspirational']"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"['adulthood', 'success', 'value']"
6,“It is better to be hated for what you are tha...,André Gide,"['life', 'love']"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"['edison', 'failure', 'inspirational', 'paraph..."
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,['misattributed-eleanor-roosevelt']
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"['humor', 'obvious', 'simile']"


In [6]:
df.describe()

,quote,author,tags
count,100,100,100
unique,100,50,84
top,“The world as we have created it is a process ...,Albert Einstein,['love']
freq,1,10,4


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   100 non-null    object
 1   author  100 non-null    object
 2   tags    100 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB


In [10]:
df.isnull().sum()

quote     0
author    0
tags      0
dtype: int64

In [13]:
df.tail(10)

,quote,author,tags
90,"“The truth."" Dumbledore sighed. ""It is a beaut...",J.K. Rowling,['truth']
91,“I'm the one that's got to die when it's time ...,Jimi Hendrix,"['death', 'life']"
92,“To die will be an awfully big adventure.”,J.M. Barrie,"['adventure', 'love']"
93,“It takes courage to grow up and become who yo...,E.E. Cummings,['courage']
94,“But better to get hurt by the truth than comf...,Khaled Hosseini,['life']
95,“You never really understand a person until yo...,Harper Lee,['better-life-empathy']
96,“You have to write the book that wants to be w...,Madeleine L'Engle,"['books', 'children', 'difficult', 'grown-ups'..."
97,“Never tell the truth to people who are not wo...,Mark Twain,['truth']
98,"“A person's a person, no matter how small.”",Dr. Seuss,['inspirational']
99,“... a mind needs books as a sword needs a whe...,George R.R. Martin,"['books', 'mind']"


In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df.dtypes

quote     object
author    object
tags      object
dtype: object

## 🔍 Initial Data Investigation

Before performing any cleaning, the raw dataset was inspected to understand its structure, data types, missing values, duplicates, and potential data-quality issues.

### Investigation Results

* **Columns:** `quote`, `author`, `tags`
* **Number of records:** 100
* **Data types:** All three columns are currently stored as `object`
* **Missing values:** No missing values were found
* **Duplicate rows:** No duplicate rows were found
* **Initial data-quality check:** No obvious inconsistencies or suspicious values were identified during the initial inspection.

### Next Step

The individual columns will be investigated in more detail before making any cleaning decisions, particularly the `tags` column because it contains lists of tags rather than simple text values.


In [21]:
print(df['quote'].dtypes)
print(df['author'].dtypes)
print(df['tags'].dtypes)

object
object
object


In [36]:
print(df['quote'].iloc[0])
print(df['author'].iloc[0])
print(df['tags'].iloc[0])

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Albert Einstein
['change', 'deep-thoughts', 'thinking', 'world']


In [37]:
(df['tags'].apply(len))

0     48
1     24
2     56
3     42
4     32
      ..
95    23
96    78
97     9
98    17
99    17
Name: tags, Length: 100, dtype: int64

In [39]:
import ast
ast.literal_eval(df['tags'].iloc[0])

['change', 'deep-thoughts', 'thinking', 'world']

In [42]:
type(ast.literal_eval(df['tags'].iloc[0]))

list

In [46]:
df['tags'] = (df['tags'].apply(ast.literal_eval))

In [49]:
df['tags']


0              [change, deep-thoughts, thinking, world]
1                                  [abilities, choices]
2        [inspirational, life, live, miracle, miracles]
3                    [aliteracy, books, classic, humor]
4                          [be-yourself, inspirational]
                            ...                        
95                                [better-life-empathy]
96    [books, children, difficult, grown-ups, write,...
97                                              [truth]
98                                      [inspirational]
99                                        [books, mind]
Name: tags, Length: 100, dtype: object

In [62]:
tags_count = (df['tags'].apply(len))

In [63]:
print(tags_count.max())

8


In [64]:
print(tags_count.min())

0


In [65]:
df.loc[tags_count == 0]

,quote,author,tags
27,“It is impossible to live without failing at s...,J.K. Rowling,[]
42,“You believe lies so you eventually learn to t...,Marilyn Monroe,[]
78,“The question isn't who is going to let me; it...,Ayn Rand,[]


### Tags Investigation

The `tags` column initially appeared as a string after loading the CSV because CSV does not preserve Python list objects. The values were converted back into lists using `ast.literal_eval()`.

Further investigation showed:

* The `tags` column contains lists after conversion.
* Most quotes have one or more tags.
* 3 quotes have an empty tag list (`[]`).
* These occur at indices 27, 42, and 78.
* Empty tag lists are treated as valid values because they indicate that no tags were assigned to those quotes rather than missing or corrupted records.
* Therefore, these rows will **not be removed** during cleaning.


## Quotes Investigation

In [66]:
df['quote'].duplicated().sum()

np.int64(0)

In [67]:
string_count = (df['quote'].apply(len))

In [72]:
df['quote'].isnull() == True

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Name: quote, Length: 100, dtype: bool

In [ ]:
string_count

0     115
1      85
2     131
3     104
4     111
     ... 
95    148
96    139
97     58
98     43
99     81
Name: quote, Length: 100, dtype: int64

In [76]:
df['quote'] = df['quote'].str.strip()

In [77]:
df['quote']

0     “The world as we have created it is a process ...
1     “It is our choices, Harry, that show what we t...
2     “There are only two ways to live your life. On...
3     “The person, be it gentleman or lady, who has ...
4     “Imperfection is beauty, madness is genius and...
                            ...                        
95    “You never really understand a person until yo...
96    “You have to write the book that wants to be w...
97    “Never tell the truth to people who are not wo...
98          “A person's a person, no matter how small.”
99    “... a mind needs books as a sword needs a whe...
Name: quote, Length: 100, dtype: object

## Author Investigate

In [78]:
df['author'].isnull().sum()

np.int64(0)

In [80]:
print(df['author'].duplicated().sum())

50


In [81]:
df['author'].nunique()

50

In [89]:
df['author'].value_counts()

author
Albert Einstein           10
J.K. Rowling               9
Marilyn Monroe             7
Dr. Seuss                  6
Mark Twain                 6
Jane Austen                5
C.S. Lewis                 5
Bob Marley                 3
Mother Teresa              2
Eleanor Roosevelt          2
Charles Bukowski           2
Ernest Hemingway           2
George R.R. Martin         2
Suzanne Collins            2
Ralph Waldo Emerson        2
Steve Martin               1
André Gide                 1
Thomas A. Edison           1
Allen Saunders             1
Pablo Neruda               1
Elie Wiesel                1
Friedrich Nietzsche        1
Douglas Adams              1
Charles M. Schulz          1
William Nicholson          1
Garrison Keillor           1
Jorge Luis Borges          1
Martin Luther King Jr.     1
Haruki Murakami            1
James Baldwin              1
Alexandre Dumas fils       1
Stephenie Meyer            1
George Eliot               1
Jim Henson                 1
George 

In [82]:
df['author'].dtypes

dtype('O')

In [87]:
df['author'].str.strip()

0        Albert Einstein
1           J.K. Rowling
2        Albert Einstein
3            Jane Austen
4         Marilyn Monroe
             ...        
95            Harper Lee
96     Madeleine L'Engle
97            Mark Twain
98             Dr. Seuss
99    George R.R. Martin
Name: author, Length: 100, dtype: object

In [88]:
(df['author'] != df['author'].str.strip()).sum()

np.int64(0)

In [90]:
df['author'].unique()

array(['Albert Einstein', 'J.K. Rowling', 'Jane Austen', 'Marilyn Monroe',
       'André Gide', 'Thomas A. Edison', 'Eleanor Roosevelt',
       'Steve Martin', 'Bob Marley', 'Dr. Seuss', 'Douglas Adams',
       'Elie Wiesel', 'Friedrich Nietzsche', 'Mark Twain',
       'Allen Saunders', 'Pablo Neruda', 'Ralph Waldo Emerson',
       'Mother Teresa', 'Garrison Keillor', 'Jim Henson',
       'Charles M. Schulz', 'William Nicholson', 'Jorge Luis Borges',
       'George Eliot', 'George R.R. Martin', 'C.S. Lewis',
       'Martin Luther King Jr.', 'James Baldwin', 'Haruki Murakami',
       'Alexandre Dumas fils', 'Stephenie Meyer', 'Ernest Hemingway',
       'Helen Keller', 'George Bernard Shaw', 'Charles Bukowski',
       'Suzanne Collins', 'J.R.R. Tolkien', 'Alfred Tennyson',
       'Terry Pratchett', 'J.D. Salinger', 'George Carlin', 'John Lennon',
       'W.C. Fields', 'Ayn Rand', 'Jimi Hendrix', 'J.M. Barrie',
       'E.E. Cummings', 'Khaled Hosseini', 'Harper Lee',
       "Madeleine L'E

## Tag Investigation

In [ ]:
df['tags'].explode()

0            change
0     deep-thoughts
0          thinking
0             world
1         abilities
          ...      
96          writing
97            truth
98    inspirational
99            books
99             mind
Name: tags, Length: 235, dtype: object

In [95]:
df['tags'].explode().value_counts()

tags
love             14
inspirational    13
life             13
humor            12
books            11
                 ..
difficult         1
grown-ups         1
write             1
writers           1
mind              1
Name: count, Length: 137, dtype: int64

In [100]:
[ tag.strip() for tags_list in df['tags'] for tag in tags_list ]

['change',
 'deep-thoughts',
 'thinking',
 'world',
 'abilities',
 'choices',
 'inspirational',
 'life',
 'live',
 'miracle',
 'miracles',
 'aliteracy',
 'books',
 'classic',
 'humor',
 'be-yourself',
 'inspirational',
 'adulthood',
 'success',
 'value',
 'life',
 'love',
 'edison',
 'failure',
 'inspirational',
 'paraphrased',
 'misattributed-eleanor-roosevelt',
 'humor',
 'obvious',
 'simile',
 'friends',
 'heartbreak',
 'inspirational',
 'life',
 'love',
 'sisters',
 'courage',
 'friends',
 'simplicity',
 'understand',
 'love',
 'fantasy',
 'life',
 'navigation',
 'activism',
 'apathy',
 'hate',
 'indifference',
 'inspirational',
 'love',
 'opposite',
 'philosophy',
 'friendship',
 'lack-of-friendship',
 'lack-of-love',
 'love',
 'marriage',
 'unhappy-marriage',
 'books',
 'contentment',
 'friends',
 'friendship',
 'life',
 'fate',
 'life',
 'misattributed-john-lennon',
 'planning',
 'plans',
 'love',
 'poetry',
 'happiness',
 'attributed-no-source',
 'humor',
 'religion',
 'humor',

In [102]:
sum(
    tag != tag.strip()
    for tags in df['tags']
    for tag in tags
)

0

In [108]:
tags = df['tags'].iloc[0]

In [109]:
len(tags)

4

In [110]:
len(set(tags))

4

In [111]:
[ (len(tags) , len(set(tags))) for tags in df['tags'] ]

[(4, 4),
 (2, 2),
 (5, 5),
 (4, 4),
 (2, 2),
 (3, 3),
 (2, 2),
 (4, 4),
 (1, 1),
 (3, 3),
 (6, 6),
 (2, 2),
 (2, 2),
 (1, 1),
 (1, 1),
 (2, 2),
 (8, 8),
 (6, 6),
 (5, 5),
 (5, 5),
 (2, 2),
 (1, 1),
 (1, 1),
 (2, 2),
 (1, 1),
 (3, 3),
 (2, 2),
 (0, 0),
 (1, 1),
 (1, 1),
 (3, 3),
 (1, 1),
 (1, 1),
 (2, 2),
 (2, 2),
 (3, 3),
 (2, 2),
 (4, 4),
 (2, 2),
 (1, 1),
 (4, 4),
 (4, 4),
 (0, 0),
 (2, 2),
 (2, 2),
 (1, 1),
 (1, 1),
 (2, 2),
 (1, 1),
 (1, 1),
 (2, 2),
 (3, 3),
 (1, 1),
 (1, 1),
 (2, 2),
 (1, 1),
 (3, 3),
 (3, 3),
 (1, 1),
 (3, 3),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (6, 6),
 (1, 1),
 (2, 2),
 (2, 2),
 (1, 1),
 (1, 1),
 (2, 2),
 (1, 1),
 (3, 3),
 (2, 2),
 (5, 5),
 (6, 6),
 (8, 8),
 (2, 2),
 (0, 0),
 (3, 3),
 (1, 1),
 (4, 4),
 (1, 1),
 (3, 3),
 (2, 2),
 (3, 3),
 (1, 1),
 (2, 2),
 (2, 2),
 (4, 4),
 (1, 1),
 (2, 2),
 (2, 2),
 (1, 1),
 (1, 1),
 (1, 1),
 (7, 7),
 (1, 1),
 (1, 1),
 (2, 2)]

## 🔍 Column Investigation Summary

* **Quote:** No missing or duplicate quotes were found. Leading/trailing whitespace was removed.
* **Author:** 50 unique authors were identified. No missing values, extra whitespace, or obvious formatting inconsistencies were found.
* **Tags:** The column contains valid lists with 235 total tag entries and 137 unique tags. Three quotes have empty tag lists, which were treated as valid values. No whitespace or duplicate tags were found within individual quotes.

### Conclusion

The dataset contains no major data-quality issues requiring extensive cleaning. Only minor preprocessing was needed, while valid values were preserved.
